# LeetCode 681 — Next Closest Time bằng Python

Link đề: https://leetcode.com/problems/next-closest-time/ · Độ khó: **Medium** (bài **Premium**) · Chủ đề: Hash Table, String, Backtracking, Enumeration

> **Ghi chú Premium:** đây là bài trả phí của LeetCode. Phần đề trong notebook được viết theo hiểu biết về bài (và đối chiếu với lời giải tham khảo công khai), nên nếu có tài khoản Premium hãy **đối chiếu lại đề gốc**. Các ví dụ 3, 4, 5 do notebook tự đặt, đáp án đã kiểm chứng bằng code.

Skeleton LeetCode:

```python
class Solution:
    def nextClosestTime(self, time: str) -> str:
        ...
```

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: cho một giờ dạng `"HH:MM"`, dùng **lại các chữ số của chính nó** (mỗi chữ số dùng bao nhiêu lần cũng được) để ghép ra giờ hợp lệ **gần nhất sau** thời điểm hiện tại, và biết rằng nếu phải **qua nửa đêm** thì ta lấy giờ của ngày hôm sau.
2. Nắm các công cụ Python: cắt chuỗi `time[:2]`, `int()`, `divmod`, định dạng `f"{h:02d}:{m:02d}"`, `set` các ký tự, `itertools.product`, `sorted` + `bisect`, và phép **modulo `% 1440`** để xử lý vòng quanh ngày.
3. Đi từ **chạy từng phút cho tới khi gặp giờ chỉ dùng chữ số cho phép** → **liệt kê mọi tổ hợp `4^4` chữ số và chọn giờ có khoảng cách vòng ngắn nhất** → **quay lui / DFS dựng từng chữ số** → **tham lam từ phải sang trái (tăng chữ số cuối có thể)** → **ghép tập giờ hợp lệ × tập phút hợp lệ, sắp xếp rồi tìm nhị phân**.
4. Hiểu mấu chốt: một ngày chỉ có **1440 phút**, nên bài toán thực chất là "tìm phần tử kế tiếp trong một tập nhỏ các mốc thời gian **xếp theo vòng tròn**" — và "gần nhất sau" nghĩa là **khoảng cách theo chiều kim đồng hồ** nhỏ nhất, **lớn hơn 0**.
5. Tránh các lỗi kinh điển: quên **vòng qua nửa đêm**, chấp nhận chính giờ hiện tại (khoảng cách `0`) thay vì `1440`, ghép ra giờ/phút **không hợp lệ** (như `19:49` hay `24:00`), và quên **đệm số 0** (`9:5` thay vì `09:05`).

## 2. Đề bài & ví dụ

Cho một thời gian `time` biểu diễn dưới dạng `"HH:MM"`. Hãy tạo ra **giờ gần nhất kế tiếp** bằng cách **dùng lại các chữ số hiện có**. Mỗi chữ số có thể dùng **lại nhiều lần**.

Bạn có thể giả sử chuỗi đầu vào luôn hợp lệ. Ví dụ `"01:34"`, `"12:09"` là hợp lệ; `"1:34"`, `"12:9"` là **không** hợp lệ (phải đủ hai chữ số mỗi phần).

**Ví dụ 1**

- Input: `time = "19:34"`
- Output: `"19:39"`
- Giải thích: các chữ số là `1, 9, 3, 4`. Giờ gần nhất tiếp theo dùng các chữ số này là `19:39` (sau 5 phút). Không phải `19:33` vì giờ đó phải chờ `23` giờ `59` phút nữa.

**Ví dụ 2**

- Input: `time = "23:59"`
- Output: `"22:22"`
- Giải thích: các chữ số là `2, 3, 5, 9`. Không có giờ nào hợp lệ lớn hơn `23:59` trong ngày, nên ta sang **ngày hôm sau** và lấy giờ nhỏ nhất ghép được: `22:22`.

**Ví dụ 3 (tự đặt, đối chiếu lại đề gốc nếu cần)**

- Input: `time = "11:11"`
- Output: `"11:11"`
- Giải thích: chỉ có một chữ số `1` nên chỉ ghép được `11:11`; giờ kế tiếp chính là `11:11` của **ngày mai** (24 giờ sau).

**Ví dụ 4 (tự đặt)**

- Input: `time = "01:32"` → Output: `"01:33"`

**Ví dụ 5 (tự đặt)**

- Input: `time = "13:55"` → Output: `"15:11"`

**Ràng buộc:**

- `time.length == 5`.
- `time` là giờ hợp lệ dạng `"HH:MM"`, `0 <= HH < 24`, `0 <= MM < 60`.

Chú ý: "kế tiếp" nghĩa là **sau** thời điểm hiện tại (khoảng cách dương). Khi chỉ ghép được đúng một giờ, đáp án là chính giờ đó (của ngày mai).

In [1]:
from typing import List
from itertools import product
from bisect import bisect_right
import random
import time as _time_module

examples = [
    ("19:34", "19:39"),
    ("23:59", "22:22"),
    ("11:11", "11:11"),
    ("01:32", "01:33"),
    ("13:55", "15:11"),
]
for s, expected in examples:
    print(repr(s), "-> kỳ vọng", repr(expected))

'19:34' -> kỳ vọng '19:39'
'23:59' -> kỳ vọng '22:22'
'11:11' -> kỳ vọng '11:11'
'01:32' -> kỳ vọng '01:33'
'13:55' -> kỳ vọng '15:11'


## 3. Kiến thức nền

### 3.1. Tách `"HH:MM"` thành số

Chuỗi `"19:34"` dài 5 ký tự: vị trí `0,1` là giờ, `2` là dấu `:`, `3,4` là phút. Cắt lát `time[:2]` lấy hai ký tự đầu, `time[3:]` lấy từ vị trí 3 đến hết. `int("09")` cho `9` (số 0 đứng đầu được bỏ qua).

In [2]:
time = "19:34"
print(time[:2], time[3:])
h, m = int(time[:2]), int(time[3:])
print("giờ =", h, "| phút =", m)
print(int("09"), int("00"), int("05"))

19 34
giờ = 19 | phút = 34
9 0 5


### 3.2. Đổi giờ ra "số phút kể từ 00:00" và ngược lại

Để so sánh giờ cho dễ, ta đổi `h:m` thành **tổng số phút** `h * 60 + m` (từ `0` đến `1439`). Đổi ngược dùng `divmod(tổng, 60)` cho `(giờ, phút)`. Định dạng `f"{h:02d}"` đệm số 0 phía trước để luôn đủ hai chữ số.

In [3]:
def to_minutes(t):
    return int(t[:2]) * 60 + int(t[3:])

def to_str(total):
    h, m = divmod(total, 60)
    return f"{h:02d}:{m:02d}"

print(to_minutes("19:34"), to_minutes("00:00"), to_minutes("23:59"))
print(to_str(1174), to_str(0), to_str(1439), to_str(65))
print(f"{9}:{5}", "<- sai định dạng;", f"{9:02d}:{5:02d}", "<- đúng")

1174 0 1439
19:34 00:00 23:59 01:05
9:5 <- sai định dạng; 09:05 <- đúng


### 3.3. Vòng quanh ngày bằng modulo `1440`

Một ngày có `24 × 60 = 1440` phút. Sau `23:59` (phút `1439`) là `00:00` (phút `0`) của hôm sau. Phép `% 1440` làm điều đó: `(1439 + 1) % 1440 == 0`. **Khoảng cách theo chiều kim đồng hồ** từ `a` đến `b` là `(b - a) % 1440` — luôn nằm trong `[0, 1439]`. Vì bài cần giờ **sau** (dương), nếu kết quả là `0` ta đổi thành `1440` (đúng một ngày sau).

In [4]:
def clockwise_gap(a, b):
    """Số phút phải chờ từ phút a tới phút b (theo chiều thời gian, có vòng nửa đêm); 0 coi là 1440."""
    d = (b - a) % 1440
    return d if d != 0 else 1440

cur = to_minutes("23:59")
for cand in ["00:00", "22:22", "23:59", "23:58"]:
    print(f"từ 23:59 tới {cand}: chờ {clockwise_gap(cur, to_minutes(cand))} phút")

từ 23:59 tới 00:00: chờ 1 phút
từ 23:59 tới 22:22: chờ 1343 phút
từ 23:59 tới 23:59: chờ 1440 phút
từ 23:59 tới 23:58: chờ 1439 phút


### 3.4. `set` các chữ số và kiểm tra "chỉ dùng chữ số cho phép"

`set("19:34")` giữ các ký tự khác nhau (`{'1','9',':','3','4'}`). Bỏ dấu `:` bằng `replace(":", "")`. Phép `A <= B` giữa hai `set` là kiểm tra **`A` là tập con của `B`** — đúng thứ ta cần: "mọi chữ số của giờ mới đều nằm trong tập cho phép".

In [5]:
allowed = set("19:34".replace(":", ""))
print(sorted(allowed))
print(set("19:39".replace(":", "")) <= allowed)     # True: chỉ dùng 1,9,3
print(set("19:35".replace(":", "")) <= allowed)     # False: có chữ số 5

['1', '3', '4', '9']
True
False


### 3.5. `itertools.product(repeat=4)`

`product(digits, repeat=4)` sinh mọi **chuỗi 4 chữ số** lấy từ `digits` (cho phép lặp): `|digits|^4` phần tử. Với tối đa 4 chữ số khác nhau, tối đa `4^4 = 256` tổ hợp — nhỏ xíu.

In [6]:
digits = sorted(set("19:34".replace(":", "")))
combos = list(product(digits, repeat=4))
print("các chữ số:", digits, "| số tổ hợp:", len(combos))
print(combos[:3], "...", combos[-2:])

các chữ số: ['1', '3', '4', '9'] | số tổ hợp: 256
[('1', '1', '1', '1'), ('1', '1', '1', '3'), ('1', '1', '1', '4')] ... [('9', '9', '9', '4'), ('9', '9', '9', '9')]


### 3.6. `sorted` và `bisect_right`

`bisect_right(a, x)` trên list `a` **đã sắp tăng** trả về vị trí chèn `x` sao cho các phần tử `<= x` nằm bên trái; vì vậy `a[bisect_right(a, x)]` là phần tử **nhỏ nhất lớn hơn hẳn `x`** (nếu còn). Rất hợp để tìm "phần tử kế tiếp".

In [7]:
a = [3, 10, 25, 40]
for x in [0, 3, 10, 24, 40, 99]:
    i = bisect_right(a, x)
    print(f"x = {x:2d}: vị trí {i} ->", a[i] if i < len(a) else "hết (quay về đầu: " + str(a[0]) + ")")

x =  0: vị trí 0 -> 3
x =  3: vị trí 1 -> 10
x = 10: vị trí 2 -> 25
x = 24: vị trí 2 -> 25
x = 40: vị trí 4 -> hết (quay về đầu: 3)
x = 99: vị trí 4 -> hết (quay về đầu: 3)


## 4. Ý tưởng nền tảng / trực giác (tính tay)

Ví dụ `19:34`, các chữ số cho phép là `{1, 3, 4, 9}`. Thử **từng phút một**, bắt đầu từ `19:35`:

- `19:35` có chữ số `5` ✗.
- `19:36`, `19:37`, `19:38` có `6`, `7`, `8` ✗.
- `19:39`: chữ số `1, 9, 3, 9` đều thuộc tập ✓ ⇒ đáp án, sau đúng 5 phút.

Ví dụ `23:59`, tập `{2, 3, 5, 9}`: không còn phút nào của ngày này lớn hơn `23:59`, nên đi qua nửa đêm. Từ `00:00` trở đi, giờ đầu tiên chỉ dùng `2, 3, 5, 9` là `22:22` (không thể là `02:xx` vì cần chữ số `0`, v.v.). Đó là giờ **nhỏ nhất** ghép được.

Ví dụ `11:11`: chỉ có chữ số `1`. Giờ ghép được duy nhất là `11:11`. Từ `11:11` đi **thêm 1440 phút** là lại `11:11` của hôm sau.

**Trực giác chính:** mọi giờ đều là một điểm trong vòng tròn `1440` điểm. Tập "giờ hợp lệ" (chỉ dùng chữ số cho phép) là một tập con của vòng tròn. Ta cần **điểm đầu tiên của tập đó nằm sau điểm hiện tại theo chiều kim đồng hồ** (có thể chính điểm hiện tại nếu đi đủ một vòng). Hãy xem trực quan: liệt kê toàn bộ giờ hợp lệ của hai ví dụ.

In [8]:
def all_valid_times(time):
    allowed = set(time.replace(":", ""))
    return [to_str(m) for m in range(1440) if set(to_str(m).replace(":", "")) <= allowed]

for s in ["19:34", "23:59", "11:11"]:
    v = all_valid_times(s)
    print(f"{s}: có {len(v):3d} giờ hợp lệ; nhỏ nhất {v[0]}, lớn nhất {v[-1]}; vài giờ đầu: {v[:6]}")

19:34: có  48 giờ hợp lệ; nhỏ nhất 11:11, lớn nhất 19:49; vài giờ đầu: ['11:11', '11:13', '11:14', '11:19', '11:31', '11:33']
23:59: có  24 giờ hợp lệ; nhỏ nhất 22:22, lớn nhất 23:59; vài giờ đầu: ['22:22', '22:23', '22:25', '22:29', '22:32', '22:33']
11:11: có   1 giờ hợp lệ; nhỏ nhất 11:11, lớn nhất 11:11; vài giờ đầu: ['11:11']


## 5. Cách 1 — Brute Force: chạy từng phút

**Ý tưởng:** làm đúng như tính tay. Bắt đầu từ phút hiện tại, **tăng từng phút** (có vòng nửa đêm), với mỗi phút kiểm tra xem mọi chữ số có thuộc tập cho phép không; gặp phút đầu tiên thỏa thì trả về. Tối đa thử `1440` phút (vì sau `1440` phút ta quay về đúng chính giờ hiện tại, luôn thỏa).

Rất dễ hiểu và chắc chắn đúng, nên ta dùng làm **oracle** để kiểm tra các cách khác (đúng cho mọi giờ, vì chỉ có 1440 giờ).

In [9]:
def next_time_bruteforce(time: str) -> str:
    allowed = set(time.replace(":", ""))
    cur = to_minutes(time)
    for step in range(1, 1441):                  # 1 .. 1440 phút sau
        s = to_str((cur + step) % 1440)
        if set(s.replace(":", "")) <= allowed:
            return s
    return time                                  # không bao giờ tới đây

for s, expected in examples:
    got = next_time_bruteforce(s)
    print(repr(s), repr(got), "OK" if got == expected else "SAI")

'19:34' '19:39' OK
'23:59' '22:22' OK
'11:11' '11:11' OK
'01:32' '01:33' OK
'13:55' '15:11' OK


**Độ phức tạp:** tối đa `1440` bước, mỗi bước tạo chuỗi và so sánh `set` cỡ `O(1)` ⇒ `O(1440)` — hằng số. Bộ nhớ `O(1)`.

**Nút thắt:** về mặt giới hạn, bài này **không có** nút thắt lớn (đầu vào luôn cỡ 5 ký tự). Nhưng cách này **đi qua cả những phút vô ích** (`19:35 ... 19:38`) và không cho thấy cấu trúc của bài. Ta muốn đi **thẳng tới các giờ hợp lệ** (chỉ tối đa `256` ứng viên), hoặc suy luận chữ số-bằng-chữ-số.

In [10]:
# Đếm số phút phải thử trong mỗi giờ vào, để thấy trường hợp xấu
worst = (0, None)
for m in range(1440):
    s = to_str(m)
    allowed = set(s.replace(":", ""))
    steps = 0
    for step in range(1, 1441):
        steps = step
        if set(to_str((m + step) % 1440).replace(":", "")) <= allowed:
            break
    if steps > worst[0]:
        worst = (steps, s)
print("giờ vào cần thử nhiều phút nhất:", worst[1], "với", worst[0], "bước")

giờ vào cần thử nhiều phút nhất: 00:00 với 1440 bước


## 6. Ý tưởng thuật toán / bước đột phá

Có **ít giờ hợp lệ** (tối đa `4^4 = 256` chuỗi 4 chữ số, chỉ một phần là giờ thật), nên thay vì đi từng phút, ta **liệt kê trực tiếp các giờ hợp lệ** rồi chọn cái "kế tiếp":

1. Lấy tập chữ số `D` của `time` (tối đa 4 chữ số khác nhau).
2. Với mỗi chuỗi 4 chữ số `abcd` lấy từ `D`: giờ `h = ab`, phút `m = cd`. Hợp lệ khi `h < 24` và `m < 60`.
3. Mỗi giờ hợp lệ là một điểm `h·60 + m` trên vòng tròn 1440. Với giờ hiện tại `cur`, khoảng cách tới điểm đó là `(điểm - cur) % 1440`, **đổi `0` thành `1440`**. Giờ cần tìm là điểm có khoảng cách **nhỏ nhất**.

Quan trọng: **đổi `0` thành `1440`** mô tả đúng "phải là giờ sau, không phải bây giờ" — và nó cho đáp án **chính giờ hiện tại** khi không còn giờ hợp lệ nào khác (như `11:11`).

Từ ý tưởng này nảy sinh ba hướng: dựng chữ số bằng **DFS**, tách giờ/phút để **sắp xếp + tìm nhị phân**, hoặc **tham lam từ phải sang trái** (tăng chữ số cuối cùng có thể mà không phá tính hợp lệ).

## 7. Các cách tiếp theo

Mỗi cách dưới đây đều nhắm tới đáp án giống nhau nhưng nhìn bài toán khác nhau. Khi nộp LeetCode, chỉ đổi tên class thành `Solution`.

### 7.1. Cách 2 — Liệt kê `4^4` tổ hợp, chọn khoảng cách vòng nhỏ nhất

**Giải thích từng đoạn:**

1. `digits = set(time) - {":"}` là tập chữ số cho phép.
2. `cur = to_minutes(time)`.
3. Duyệt `product(digits, repeat=4)`; ghép `h = int(a + b)`, `m = int(c + d)`; bỏ nếu `h >= 24` hoặc `m >= 60`.
4. Với mỗi giờ hợp lệ, tính khoảng cách `d = (t - cur) % 1440`, đổi `0` thành `1440`; giữ giờ có `d` nhỏ nhất.

In [11]:
class SolutionProduct:
    def nextClosestTime(self, time: str) -> str:
        digits = set(time) - {":"}
        cur = int(time[:2]) * 60 + int(time[3:])
        best_gap, best = 1441, time
        for a, b, c, d in product(digits, repeat=4):
            h, m = int(a + b), int(c + d)
            if h >= 24 or m >= 60:
                continue
            gap = (h * 60 + m - cur) % 1440 or 1440
            if gap < best_gap:
                best_gap, best = gap, f"{h:02d}:{m:02d}"
        return best

for s, expected in examples:
    print(repr(s), repr(SolutionProduct().nextClosestTime(s)), "kỳ vọng", repr(expected))

'19:34' '19:39' kỳ vọng '19:39'
'23:59' '22:22' kỳ vọng '22:22'
'11:11' '11:11' kỳ vọng '11:11'
'01:32' '01:33' kỳ vọng '01:33'
'13:55' '15:11' kỳ vọng '15:11'


**Code nộp LeetCode (Cách 2):**

```python
class Solution:
    def nextClosestTime(self, time: str) -> str:
        digits = set(time) - {":"}
        cur = int(time[:2]) * 60 + int(time[3:])
        best_gap, best = 1441, time
        for a, b, c, d in product(digits, repeat=4):   # from itertools import product
            h, m = int(a + b), int(c + d)
            if h >= 24 or m >= 60:
                continue
            gap = (h * 60 + m - cur) % 1440 or 1440
            if gap < best_gap:
                best_gap, best = gap, f"{h:02d}:{m:02d}"
        return best
```

**Độ phức tạp:** tối đa `4^4 = 256` tổ hợp, mỗi tổ hợp `O(1)` ⇒ `O(1)` thời gian (hằng số nhỏ); bộ nhớ `O(1)`. Tổng quát với `k` chữ số trong tập và `L` vị trí: `O(k^L)`.

### 7.2. Cách 3 — Quay lui / DFS dựng từng chữ số

**Ý tưởng:** thay vì `product`, ta tự dựng chuỗi 4 chữ số bằng **đệ quy**: mỗi bước thử từng chữ số cho vị trí kế tiếp. Ưu điểm: **cắt tỉa sớm** — nếu hai chữ số đầu đã tạo giờ `>= 24` thì khỏi thử tiếp; nếu chữ số phút hàng chục `> 5` cũng bỏ ngay.

**Giải thích từng đoạn:**

1. `dfs(prefix)`: `prefix` là chuỗi chữ số đã chọn (độ dài 0..4).
2. Khi `len(prefix) == 2` kiểm tra giờ `< 24`; khi `len(prefix) == 3` kiểm tra chữ số phút hàng chục `<= 5`.
3. Khi đủ 4 chữ số, tính khoảng cách vòng và cập nhật kết quả tốt nhất.

In [12]:
class SolutionDFS:
    def nextClosestTime(self, time: str) -> str:
        digits = sorted(set(time) - {":"})
        cur = int(time[:2]) * 60 + int(time[3:])
        best = [1441, time]

        def dfs(prefix: str) -> None:
            if len(prefix) == 2 and int(prefix) >= 24:       # giờ không hợp lệ: cắt tỉa
                return
            if len(prefix) == 3 and int(prefix[2]) > 5:      # phút hàng chục > 5: cắt tỉa
                return
            if len(prefix) == 4:
                h, m = int(prefix[:2]), int(prefix[2:])
                gap = (h * 60 + m - cur) % 1440 or 1440
                if gap < best[0]:
                    best[0], best[1] = gap, f"{prefix[:2]}:{prefix[2:]}"
                return
            for d in digits:
                dfs(prefix + d)

        dfs("")
        return best[1]

for s, expected in examples:
    print(repr(s), repr(SolutionDFS().nextClosestTime(s)), "kỳ vọng", repr(expected))

'19:34' '19:39' kỳ vọng '19:39'
'23:59' '22:22' kỳ vọng '22:22'
'11:11' '11:11' kỳ vọng '11:11'
'01:32' '01:33' kỳ vọng '01:33'
'13:55' '15:11' kỳ vọng '15:11'


**Code nộp LeetCode (Cách 3):**

```python
class Solution:
    def nextClosestTime(self, time: str) -> str:
        digits = sorted(set(time) - {":"})
        cur = int(time[:2]) * 60 + int(time[3:])
        best = [1441, time]

        def dfs(prefix: str) -> None:
            if len(prefix) == 2 and int(prefix) >= 24:
                return
            if len(prefix) == 3 and int(prefix[2]) > 5:
                return
            if len(prefix) == 4:
                h, m = int(prefix[:2]), int(prefix[2:])
                gap = (h * 60 + m - cur) % 1440 or 1440
                if gap < best[0]:
                    best[0], best[1] = gap, f"{prefix[:2]}:{prefix[2:]}"
                return
            for d in digits:
                dfs(prefix + d)

        dfs("")
        return best[1]
```

**Độ phức tạp:** số nút của cây quay lui tối đa `1 + k + k² + k³ + k⁴` với `k ≤ 4`, tức `O(1)`; nhờ cắt tỉa thực tế ít hơn. Bộ nhớ: ngăn xếp đệ quy sâu 4.

### 7.3. Cách 4 — Tham lam từ phải sang trái

**Ý tưởng khác hẳn:** ta muốn giờ **lớn hơn hiện tại nhưng nhỏ nhất có thể**. Với số có nhiều chữ số, cách làm quen thuộc là **tăng chữ số bên phải nhất có thể tăng**, rồi **đặt mọi chữ số bên phải nó về giá trị nhỏ nhất**. Ở đây "có thể tăng" nghĩa là **tồn tại chữ số cho phép lớn hơn mà vẫn hợp lệ**:

1. **Chữ số cuối (phút hàng đơn vị):** tìm chữ số cho phép **lớn hơn** nó (mọi chữ số `0..9` đều hợp lệ ở vị trí này). Nếu có, đặt và xong.
2. **Phút hàng chục:** tìm chữ số cho phép lớn hơn nó và `<= 5`. Nếu có, đặt, rồi đặt chữ số cuối về **chữ số nhỏ nhất**.
3. **Giờ hàng đơn vị:** tìm chữ số cho phép lớn hơn nó, và nếu giờ hàng chục là `2` thì phải `<= 3`. Nếu có, đặt, rồi đặt hai chữ số phút về nhỏ nhất.
4. **Giờ hàng chục:** tìm chữ số cho phép lớn hơn nó và `<= 2`. Nếu có, đặt, rồi đặt ba chữ số sau về nhỏ nhất.
5. Không tăng được ở đâu cả ⇒ **qua nửa đêm**: cả bốn chữ số là chữ số nhỏ nhất.

Vì sao đặt các chữ số bên phải về **nhỏ nhất** vẫn hợp lệ? Chữ số nhỏ nhất `lo` nhỏ hơn hay bằng **mọi** chữ số cho phép, trong đó có chữ số đang đứng ở vị trí cần "nhỏ", nên nếu vị trí đó cần `<= 5` (phút chục) thì `lo <= 5`; nếu cần "giờ hợp lệ" thì `lo` không làm giờ vượt giới hạn.

In [13]:
class SolutionGreedy:
    def nextClosestTime(self, time: str) -> str:
        digits = sorted(set(time) - {":"})
        lo = digits[0]
        t = [time[0], time[1], time[3], time[4]]        # bỏ dấu ":" để đánh số 0..3

        def bigger(c, limit):
            """Chữ số cho phép nhỏ nhất lớn hơn c và <= limit; không có thì None."""
            for d in digits:
                if d > c and d <= limit:
                    return d
            return None

        d = bigger(t[3], "9")                             # phút hàng đơn vị
        if d:
            t[3] = d
        else:
            d = bigger(t[2], "5")                         # phút hàng chục
            if d:
                t[2], t[3] = d, lo
            else:
                d = bigger(t[1], "3" if t[0] == "2" else "9")   # giờ hàng đơn vị
                if d:
                    t[1], t[2], t[3] = d, lo, lo
                else:
                    d = bigger(t[0], "2")                 # giờ hàng chục
                    if d:
                        t[0], t[1], t[2], t[3] = d, lo, lo, lo
                    else:
                        t = [lo, lo, lo, lo]              # qua nửa đêm
        return f"{t[0]}{t[1]}:{t[2]}{t[3]}"

for s, expected in examples:
    print(repr(s), repr(SolutionGreedy().nextClosestTime(s)), "kỳ vọng", repr(expected))

'19:34' '19:39' kỳ vọng '19:39'
'23:59' '22:22' kỳ vọng '22:22'
'11:11' '11:11' kỳ vọng '11:11'
'01:32' '01:33' kỳ vọng '01:33'
'13:55' '15:11' kỳ vọng '15:11'


**Code nộp LeetCode (Cách 4):**

```python
class Solution:
    def nextClosestTime(self, time: str) -> str:
        digits = sorted(set(time) - {":"})
        lo = digits[0]
        t = [time[0], time[1], time[3], time[4]]

        def bigger(c, limit):
            for d in digits:
                if d > c and d <= limit:
                    return d
            return None

        d = bigger(t[3], "9")
        if d:
            t[3] = d
        else:
            d = bigger(t[2], "5")
            if d:
                t[2], t[3] = d, lo
            else:
                d = bigger(t[1], "3" if t[0] == "2" else "9")
                if d:
                    t[1], t[2], t[3] = d, lo, lo
                else:
                    d = bigger(t[0], "2")
                    if d:
                        t[0], t[1], t[2], t[3] = d, lo, lo, lo
                    else:
                        t = [lo, lo, lo, lo]
        return f"{t[0]}{t[1]}:{t[2]}{t[3]}"
```

**Độ phức tạp:** thời gian `O(1)` (tối đa 4 lần quét tập ≤ 4 chữ số), bộ nhớ `O(1)`. Đây là cách nhanh nhất nhưng **dễ sai nhất** vì phải nhớ từng ngưỡng (`<= 5`, `<= 3`, `<= 2`).

Có một chi tiết cần kiểm: khi tăng **giờ hàng chục** lên `c`, ta đặt giờ hàng đơn vị về `lo`; giờ `c·lo` hợp lệ chứ? `lo <= c` vì `lo` nhỏ nhất, mà `c <= 2`, nên `lo <= 2` và `c·lo <= 22 < 24` ✓. Ta cũng sẽ kiểm tra toàn diện ở mục 12 (đối chiếu cả 1440 giờ).

### 7.4. Cách 5 — Tập giờ × tập phút, sắp xếp rồi tìm nhị phân

**Ý tưởng:** giờ và phút **độc lập** về mặt ràng buộc: tập giờ hợp lệ `H` (các cặp chữ số `< 24`) và tập phút hợp lệ `M` (các cặp `< 60`). Mọi giờ hợp lệ là `h·60 + m` với `h ∈ H`, `m ∈ M`. Ta sinh **toàn bộ mốc** này, **sắp xếp** thành list `times`, rồi dùng **tìm nhị phân** `bisect_right(times, cur)` để lấy mốc nhỏ nhất **lớn hơn hẳn** `cur`; nếu không còn mốc nào thì **quay về phần tử đầu** (ngày hôm sau).

**Giải thích từng đoạn:**

1. `pairs = {a + b for a, b in product(digits, repeat=2)}` — các chuỗi hai chữ số.
2. `hours = [int(p) for p in pairs if int(p) < 24]`, `minutes = [int(p) for p in pairs if int(p) < 60]`.
3. `times = sorted(h * 60 + m for h in hours for m in minutes)`.
4. `i = bisect_right(times, cur)`; đáp án `times[i]` nếu `i < len(times)`, ngược lại `times[0]`.

In [14]:
class SolutionBisect:
    def nextClosestTime(self, time: str) -> str:
        digits = set(time) - {":"}
        cur = int(time[:2]) * 60 + int(time[3:])
        pairs = {a + b for a, b in product(digits, repeat=2)}
        hours = [int(p) for p in pairs if int(p) < 24]
        minutes = [int(p) for p in pairs if int(p) < 60]
        times = sorted(h * 60 + m for h in hours for m in minutes)
        i = bisect_right(times, cur)
        nxt = times[i] if i < len(times) else times[0]
        return f"{nxt // 60:02d}:{nxt % 60:02d}"

for s, expected in examples:
    print(repr(s), repr(SolutionBisect().nextClosestTime(s)), "kỳ vọng", repr(expected))

'19:34' '19:39' kỳ vọng '19:39'
'23:59' '22:22' kỳ vọng '22:22'
'11:11' '11:11' kỳ vọng '11:11'
'01:32' '01:33' kỳ vọng '01:33'
'13:55' '15:11' kỳ vọng '15:11'


**Code nộp LeetCode (Cách 5):**

```python
from bisect import bisect_right
from itertools import product

class Solution:
    def nextClosestTime(self, time: str) -> str:
        digits = set(time) - {":"}
        cur = int(time[:2]) * 60 + int(time[3:])
        pairs = {a + b for a, b in product(digits, repeat=2)}
        hours = [int(p) for p in pairs if int(p) < 24]
        minutes = [int(p) for p in pairs if int(p) < 60]
        times = sorted(h * 60 + m for h in hours for m in minutes)
        i = bisect_right(times, cur)
        nxt = times[i] if i < len(times) else times[0]
        return f"{nxt // 60:02d}:{nxt % 60:02d}"
```

**Độ phức tạp:** `pairs` tối đa 16 phần tử; `times` tối đa `16 × 16 = 256` mốc; sắp xếp `O(256 log 256)`; tìm nhị phân `O(log 256)` ⇒ `O(1)` (hằng số nhỏ), bộ nhớ `O(1)`.

Điểm hay của cách này: **không cần xét modulo thủ công** — `bisect_right` cho "kế tiếp trong ngày", còn phép "quay về `times[0]`" là cách mô tả vòng qua nửa đêm. Khi `times` chỉ có một phần tử (như `11:11`), `bisect_right` trả `1` (hết list) nên ta quay về `times[0]`, đúng chính giờ đó.

## 8. TRACE từng bước

Ta theo dõi cách 4 (tham lam từ phải sang trái) trên `19:34` và `23:59`: in tập chữ số, từng vị trí đang xét, chữ số lớn hơn tìm được (hoặc không), và chuỗi sau mỗi bước.

Bảng trạng thái cho `19:34` (chữ số `{1, 3, 4, 9}`):

| bước | vị trí | giá trị hiện tại | giới hạn | chữ số lớn hơn tìm được | kết quả |
|---|---|---|---|---|---|
| 1 | phút đơn vị | `4` | `<= 9` | `9` | đặt `9` ⇒ `19:39`, dừng |

Bảng cho `23:59` (chữ số `{2, 3, 5, 9}`):

| bước | vị trí | giá trị hiện tại | giới hạn | chữ số lớn hơn tìm được | kết quả |
|---|---|---|---|---|---|
| 1 | phút đơn vị | `9` | `<= 9` | không | tiếp tục |
| 2 | phút chục | `5` | `<= 5` | không (`9 > 5`) | tiếp tục |
| 3 | giờ đơn vị | `3` | `<= 3` (vì giờ chục là `2`) | không | tiếp tục |
| 4 | giờ chục | `2` | `<= 2` | không (`3, 5, 9 > 2`) | tiếp tục |
| 5 | — | — | — | — | qua nửa đêm ⇒ `22:22` |

In [15]:
def trace_greedy(time):
    digits = sorted(set(time) - {":"})
    lo = digits[0]
    print(f"time = {time} | chữ số cho phép = {digits} | nhỏ nhất = {lo!r}")
    t = [time[0], time[1], time[3], time[4]]
    names = ["giờ chục", "giờ đơn vị", "phút chục", "phút đơn vị"]

    def bigger(c, limit):
        for d in digits:
            if d > c and d <= limit:
                return d
        return None

    steps = [(3, lambda: "9"), (2, lambda: "5"), (1, lambda: "3" if t[0] == "2" else "9"), (0, lambda: "2")]
    for pos, limit_fn in steps:
        limit = limit_fn()
        d = bigger(t[pos], limit)
        print(f"  xét {names[pos]:11s}: hiện {t[pos]!r}, giới hạn <= {limit!r}, chữ số lớn hơn: {d!r}")
        if d:
            t[pos] = d
            for q in range(pos + 1, 4):
                t[q] = lo
            res = f"{t[0]}{t[1]}:{t[2]}{t[3]}"
            print(f"  -> đặt {d!r} ở {names[pos]}, các vị trí bên phải về {lo!r}: {res}")
            return res
    res = lo * 2 + ":" + lo * 2
    print(f"  -> không tăng được ở đâu: qua nửa đêm, lấy {res}")
    return res

print("KẾT QUẢ:", trace_greedy("19:34"))
print()
print("KẾT QUẢ:", trace_greedy("23:59"))
print()
print("KẾT QUẢ:", trace_greedy("13:55"))

time = 19:34 | chữ số cho phép = ['1', '3', '4', '9'] | nhỏ nhất = '1'
  xét phút đơn vị: hiện '4', giới hạn <= '9', chữ số lớn hơn: '9'
  -> đặt '9' ở phút đơn vị, các vị trí bên phải về '1': 19:39
KẾT QUẢ: 19:39

time = 23:59 | chữ số cho phép = ['2', '3', '5', '9'] | nhỏ nhất = '2'
  xét phút đơn vị: hiện '9', giới hạn <= '9', chữ số lớn hơn: None
  xét phút chục  : hiện '5', giới hạn <= '5', chữ số lớn hơn: None
  xét giờ đơn vị : hiện '3', giới hạn <= '3', chữ số lớn hơn: None
  xét giờ chục   : hiện '2', giới hạn <= '2', chữ số lớn hơn: None
  -> không tăng được ở đâu: qua nửa đêm, lấy 22:22
KẾT QUẢ: 22:22

time = 13:55 | chữ số cho phép = ['1', '3', '5'] | nhỏ nhất = '1'
  xét phút đơn vị: hiện '5', giới hạn <= '9', chữ số lớn hơn: None
  xét phút chục  : hiện '5', giới hạn <= '5', chữ số lớn hơn: None
  xét giờ đơn vị : hiện '3', giới hạn <= '9', chữ số lớn hơn: '5'
  -> đặt '5' ở giờ đơn vị, các vị trí bên phải về '1': 15:11
KẾT QUẢ: 15:11


Giờ trace cách 5: in tập giờ, tập phút, vài mốc đầu của `times`, vị trí `bisect_right` và đáp án.

In [16]:
def trace_bisect(time):
    digits = set(time) - {":"}
    cur = to_minutes(time)
    pairs = {a + b for a, b in product(digits, repeat=2)}
    hours = sorted(int(p) for p in pairs if int(p) < 24)
    minutes = sorted(int(p) for p in pairs if int(p) < 60)
    times = sorted(h * 60 + m for h in hours for m in minutes)
    i = bisect_right(times, cur)
    print(f"time = {time} (= {cur} phút) | chữ số = {sorted(digits)}")
    print("  giờ hợp lệ  :", hours)
    print("  phút hợp lệ :", minutes)
    print("  số mốc      :", len(times), "| mốc đầu:", [to_str(x) for x in times[:5]], "...")
    print(f"  bisect_right(times, {cur}) = {i}", "-> còn mốc sau" if i < len(times) else "-> hết, quay về times[0]")
    nxt = times[i] if i < len(times) else times[0]
    return to_str(nxt)

print("KẾT QUẢ:", trace_bisect("19:34"))
print()
print("KẾT QUẢ:", trace_bisect("23:59"))

time = 19:34 (= 1174 phút) | chữ số = ['1', '3', '4', '9']
  giờ hợp lệ  : [11, 13, 14, 19]
  phút hợp lệ : [11, 13, 14, 19, 31, 33, 34, 39, 41, 43, 44, 49]
  số mốc      : 48 | mốc đầu: ['11:11', '11:13', '11:14', '11:19', '11:31'] ...
  bisect_right(times, 1174) = 43 -> còn mốc sau
KẾT QUẢ: 19:39

time = 23:59 (= 1439 phút) | chữ số = ['2', '3', '5', '9']
  giờ hợp lệ  : [22, 23]
  phút hợp lệ : [22, 23, 25, 29, 32, 33, 35, 39, 52, 53, 55, 59]
  số mốc      : 24 | mốc đầu: ['22:22', '22:23', '22:25', '22:29', '22:32'] ...
  bisect_right(times, 1439) = 24 -> hết, quay về times[0]
KẾT QUẢ: 22:22


## 9. Code nộp LeetCode

Ta chọn **Cách 5 (tập giờ × tập phút, sắp xếp, tìm nhị phân)** làm bản nộp: ngắn, **không có ngưỡng ma thuật**, và xử lý tự nhiên cả trường hợp vòng qua nửa đêm. Class `Solution` đúng chữ ký:

```python
from bisect import bisect_right
from itertools import product

class Solution:
    def nextClosestTime(self, time: str) -> str:
        digits = set(time) - {":"}
        cur = int(time[:2]) * 60 + int(time[3:])
        pairs = {a + b for a, b in product(digits, repeat=2)}
        hours = [int(p) for p in pairs if int(p) < 24]
        minutes = [int(p) for p in pairs if int(p) < 60]
        times = sorted(h * 60 + m for h in hours for m in minutes)
        i = bisect_right(times, cur)
        nxt = times[i] if i < len(times) else times[0]
        return f"{nxt // 60:02d}:{nxt % 60:02d}"
```

In [17]:
class Solution:
    def nextClosestTime(self, time: str) -> str:
        digits = set(time) - {":"}
        cur = int(time[:2]) * 60 + int(time[3:])
        pairs = {a + b for a, b in product(digits, repeat=2)}
        hours = [int(p) for p in pairs if int(p) < 24]
        minutes = [int(p) for p in pairs if int(p) < 60]
        times = sorted(h * 60 + m for h in hours for m in minutes)
        i = bisect_right(times, cur)
        nxt = times[i] if i < len(times) else times[0]
        return f"{nxt // 60:02d}:{nxt % 60:02d}"

for s, expected in examples:
    got = Solution().nextClosestTime(s)
    assert got == expected, (s, got, expected)
    print(repr(s), "->", repr(got), "OK")

'19:34' -> '19:39' OK
'23:59' -> '22:22' OK
'11:11' -> '11:11' OK
'01:32' -> '01:33' OK
'13:55' -> '15:11' OK


## 10. Vì sao đúng

**Tính đầy đủ của tập ứng viên.** Một giờ hợp lệ là `HH:MM` với `0 <= HH < 24`, `0 <= MM < 60`, mọi chữ số thuộc `D` (tập chữ số của `time`). `HH` là hai chữ số `ab` với `ab < 24` và `a, b ∈ D`; `MM` là hai chữ số `cd` với `cd < 60`, `c, d ∈ D`. Cách 5 sinh **đúng** tập `{h·60 + m : h ∈ H, m ∈ M}` — tất cả các giờ hợp lệ và không thêm giờ nào ngoài.

**Tính đúng của "kế tiếp".** Gọi `cur` là phút hiện tại và `S` là tập giờ hợp lệ (luôn chứa `cur` vì `time` dùng chính các chữ số của nó). Theo chiều kim đồng hồ, giờ gần nhất **sau** `cur` là phần tử của `S` có khoảng cách `(t - cur) mod 1440` nhỏ nhất **và dương** (khoảng cách `0` được tính là `1440`). Có hai trường hợp:

- Tồn tại `t ∈ S` với `t > cur`: mọi `t' > cur` có khoảng cách `t' - cur`; nhỏ nhất ứng với `t'` nhỏ nhất lớn hơn `cur` — chính là `times[bisect_right(times, cur)]`. Các `t'' < cur` có khoảng cách `t'' - cur + 1440 > 1440 - cur`... nhưng luôn **lớn hơn** khoảng cách của bất kỳ `t' > cur` (vì `t' - cur <= 1439 - cur < t'' - cur + 1440`). Vậy chọn đúng.
- Không có `t > cur`: mọi phần tử của `S` đều `<= cur`; khoảng cách của `t` là `t - cur + 1440` (hoặc `1440` nếu `t = cur`), nhỏ nhất khi `t` **nhỏ nhất** ⇒ `times[0]`. Cách 5 quay về `times[0]`. Đúng.

**Vì sao tham lam (cách 4) đúng.** Với chuỗi 4 chữ số `t0 t1 t2 t3` có "trọng số" giảm dần từ trái sang phải, giờ **tiếp theo lớn hơn** (theo thứ tự từ điển, cùng độ dài) có dạng: giữ nguyên tiền tố, tăng **vị trí phải nhất có thể tăng** lên chữ số hợp lệ nhỏ nhất lớn hơn, và đặt phần đuôi nhỏ nhất. Vì chuỗi `HH:MM` hợp lệ có thứ tự từ điển trùng với thứ tự thời gian, đó chính là giờ kế tiếp.

**Kiểm chứng bằng code:** cả năm cách phải cho cùng kết quả với oracle chạy từng phút trên **toàn bộ 1440 giờ trong ngày** — cell dưới. (Đây là kiểm tra vét cạn đầy đủ, mạnh hơn mọi lập luận.)

In [18]:
all_times = [to_str(m) for m in range(1440)]
impls = {
    "product": lambda s: SolutionProduct().nextClosestTime(s),
    "dfs": lambda s: SolutionDFS().nextClosestTime(s),
    "greedy": lambda s: SolutionGreedy().nextClosestTime(s),
    "bisect": lambda s: SolutionBisect().nextClosestTime(s),
    "final": lambda s: Solution().nextClosestTime(s),
}
for name, fn in impls.items():
    bad = [s for s in all_times if fn(s) != next_time_bruteforce(s)]
    print(f"{name:8s}: đúng {1440 - len(bad)}/1440 giờ", "" if not bad else f"| ví dụ sai: {bad[:3]}")
    assert not bad

product : đúng 1440/1440 giờ 


dfs     : đúng 1440/1440 giờ 
greedy  : đúng 1440/1440 giờ 


bisect  : đúng 1440/1440 giờ 
final   : đúng 1440/1440 giờ 


## 11. Time / Space Complexity

Đầu vào luôn có độ dài `5`, nên mọi cách đều `O(1)`. Điều cần so sánh là **hằng số** và khả năng **mở rộng** (nếu giờ có nhiều chữ số hơn, ví dụ `HH:MM:SS`).

| Cách | Số phép chính (bài này) | Time | Space | Mở rộng với `L` vị trí, `k` chữ số |
|---|---|---|---|---|
| 1. Chạy từng phút | tối đa 1440 bước | `O(1440)` | `O(1)` | `O(số mốc trong ngày)` |
| 2. `product` | `<= 256` tổ hợp | `O(1)` | `O(1)` | `O(k^L)` |
| 3. DFS cắt tỉa | `<= 341` nút | `O(1)` | `O(L)` | `O(k^L)` nhưng cắt tỉa |
| 4. Tham lam | `<= 4` lần quét | `O(1)` | `O(1)` | `O(L · k)` |
| 5. Giờ × phút + `bisect` | `<= 256` mốc, sắp xếp | `O(1)` | `O(1)` | `O(k^L log)` |

Đo thực tế trên toàn bộ 1440 giờ: mỗi cách chạy bao nhiêu mili giây tổng cộng.

In [19]:
for name, fn in [("brute (từng phút)", next_time_bruteforce), *impls.items()]:
    t0 = _time_module.perf_counter()
    for s in all_times:
        fn(s)
    dt = _time_module.perf_counter() - t0
    print(f"{name:18s} {dt * 1000:8.1f} ms cho 1440 giờ")

brute (từng phút)     173.5 ms cho 1440 giờ
product                58.7 ms cho 1440 giờ
dfs                    45.9 ms cho 1440 giờ
greedy                  2.1 ms cho 1440 giờ
bisect                 18.3 ms cho 1440 giờ
final                  19.1 ms cho 1440 giờ


## 12. Bộ test

Vì chỉ có **1440 giờ** khả dĩ, ta kiểm **toàn bộ không gian đầu vào** bằng oracle độc lập (chạy từng phút). Ngoài ra: ví dụ đề, edge case, các tính chất bất biến (đáp án chỉ dùng chữ số của đầu vào; khoảng cách trong `(0, 1440]`; định dạng đúng), phản ví dụ, và lấy mẫu ngẫu nhiên có seed.

In [20]:
def check_all(s, expected=None):
    results = {name: fn(s) for name, fn in impls.items()}
    results["brute"] = next_time_bruteforce(s)
    assert len(set(results.values())) == 1, (s, results)
    r = results["final"]
    if expected is not None:
        assert r == expected, (s, r, expected)
    return r

# 1) Ví dụ đề
for s, expected in examples:
    check_all(s, expected)
print("ví dụ đề: OK")

# 2) Edge case
edge = [
    ("00:00", "00:00"),            # chỉ chữ số 0 -> chính nó (24 giờ sau)
    ("22:22", "22:22"),
    ("11:11", "11:11"),
    ("23:59", "22:22"),            # qua nửa đêm
    ("23:23", "23:32"),
    ("00:01", "00:10"),            # chữ số 0,1
    ("01:59", "05:00"),            # chữ số 0,1,5,9
    ("12:59", "15:11"),
    ("20:48", "22:00"),
    ("19:59", "11:11"),            # sau 19:59 không còn giờ hợp lệ trong ngày
    ("13:55", "15:11"),
    ("09:59", "00:00"),            # giờ hợp lệ lớn nhất là 09 nên phải sang ngày mới
]
for s, expected in edge:
    check_all(s, expected)
print("edge case: OK")

ví dụ đề: OK
edge case: OK


In [21]:
# 3) Toàn bộ 1440 giờ: mọi cách khớp oracle (đã làm ở mục 10, làm lại qua check_all) và các tính chất bất biến
count_same = 0
for m in range(1440):
    s = to_str(m)
    r = check_all(s)
    assert len(r) == 5 and r[2] == ":" and r[:2].isdigit() and r[3:].isdigit(), r
    assert int(r[:2]) < 24 and int(r[3:]) < 60
    assert set(r.replace(":", "")) <= set(s.replace(":", "")), (s, r)           # chỉ dùng chữ số của đầu vào
    gap = clockwise_gap(m, to_minutes(r))
    assert 1 <= gap <= 1440
    count_same += (r == s)
print("1440 giờ: OK | số giờ mà đáp án chính là giờ đó của ngày mai:", count_same)

# 4) Phản ví dụ: những đáp án "trông hợp lý" nhưng sai
for s, wrong in [("19:34", "19:33"), ("23:59", "23:59"), ("01:32", "01:31")]:
    right = Solution().nextClosestTime(s)
    print(f"{s}: đáp án đúng {right} (khoảng {clockwise_gap(to_minutes(s), to_minutes(right))} phút) khác {wrong} (khoảng {clockwise_gap(to_minutes(s), to_minutes(wrong))} phút)")
    assert wrong != right

# 5) Lấy mẫu ngẫu nhiên có seed (xáo thứ tự gọi, kiểm tra tính nhất quán)
rng = random.Random(681)
sample = rng.sample(all_times, 200)
for s in sample:
    check_all(s)
print("mẫu ngẫu nhiên 200 giờ: OK")

1440 giờ: OK | số giờ mà đáp án chính là giờ đó của ngày mai: 3
19:34: đáp án đúng 19:39 (khoảng 5 phút) khác 19:33 (khoảng 1439 phút)
23:59: đáp án đúng 22:22 (khoảng 1343 phút) khác 23:59 (khoảng 1440 phút)
01:32: đáp án đúng 01:33 (khoảng 1 phút) khác 01:31 (khoảng 1439 phút)
mẫu ngẫu nhiên 200 giờ: OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1: quên vòng qua nửa đêm

Chỉ tìm giờ hợp lệ **lớn hơn** trong cùng ngày, và khi không có thì không biết làm gì (trả `None`, hoặc trả chính `time`).

In [22]:
class WrongNoWrap:
    def nextClosestTime(self, time: str) -> str:
        digits = set(time) - {":"}
        cur = int(time[:2]) * 60 + int(time[3:])
        best = None
        for a, b, c, d in product(digits, repeat=4):
            h, m = int(a + b), int(c + d)
            if h < 24 and m < 60 and h * 60 + m > cur:
                if best is None or h * 60 + m < best:
                    best = h * 60 + m
        return f"{best // 60:02d}:{best % 60:02d}" if best is not None else time       # SAI khi phải qua ngày

print("sai :", WrongNoWrap().nextClosestTime("23:59"), "(đúng phải là 22:22)")
print("đúng:", Solution().nextClosestTime("23:59"))

sai : 23:59 (đúng phải là 22:22)
đúng: 22:22


**Vì sao sai:** khi không còn giờ nào lớn hơn trong ngày, đáp án là giờ **nhỏ nhất** của ngày hôm sau. **Cách sửa:** dùng khoảng cách vòng `(t - cur) % 1440`, hoặc quay về `times[0]`.

### Lỗi 2: chấp nhận chính giờ hiện tại (khoảng cách `0`)

In [23]:
class WrongZeroGap:
    def nextClosestTime(self, time: str) -> str:
        digits = set(time) - {":"}
        cur = int(time[:2]) * 60 + int(time[3:])
        best_gap, best = 1441, time
        for a, b, c, d in product(digits, repeat=4):
            h, m = int(a + b), int(c + d)
            if h >= 24 or m >= 60:
                continue
            gap = (h * 60 + m - cur) % 1440             # SAI: gap = 0 cho chính giờ hiện tại
            if gap < best_gap:
                best_gap, best = gap, f"{h:02d}:{m:02d}"
        return best

print("sai :", WrongZeroGap().nextClosestTime("19:34"), "(đúng phải là 19:39)")
print("đúng:", Solution().nextClosestTime("19:34"))

sai : 19:34 (đúng phải là 19:39)
đúng: 19:39


**Vì sao sai:** chính `19:34` luôn ghép được và có khoảng cách `0`, nên luôn thắng. Đề yêu cầu giờ **kế tiếp** (sau hiện tại). **Cách sửa:** đổi `0` thành `1440`: `gap = (...) % 1440 or 1440`.

### Lỗi 3: ghép giờ/phút không kiểm tra hợp lệ

In [24]:
class WrongNoValidation:
    def nextClosestTime(self, time: str) -> str:
        digits = sorted(set(time) - {":"})
        cur = int(time[:2]) * 60 + int(time[3:])
        best_gap, best = 10 ** 9, time
        for a, b, c, d in product(digits, repeat=4):
            h, m = int(a + b), int(c + d)               # SAI: không kiểm h < 24, m < 60
            gap = (h * 60 + m - cur) % 1440 or 1440
            if gap < best_gap:
                best_gap, best = gap, f"{a}{b}:{c}{d}"
        return best

print("sai :", WrongNoValidation().nextClosestTime("13:59"), "(đúng phải là 15:11)")
print("đúng:", Solution().nextClosestTime("13:59"))

sai : 13:91 (đúng phải là 15:11)
đúng: 15:11


**Vì sao sai:** với `13:59`, tổ hợp `13:91` không phải giờ thật (phút `91`), nhưng `13 * 60 + 91 = 871` tương ứng `14:31`, chỉ cách `32` phút nên "thắng" giờ hợp lệ `15:11` (cách `72` phút); chuỗi in ra lại là `13:91`. **Cách sửa:** luôn kiểm `h < 24` và `m < 60` trước khi xét.

### Lỗi 4: quên đệm số 0 khi định dạng

In [25]:
class WrongNoPad:
    def nextClosestTime(self, time: str) -> str:
        digits = set(time) - {":"}
        cur = int(time[:2]) * 60 + int(time[3:])
        best_gap, best = 1441, time
        for a, b, c, d in product(digits, repeat=4):
            h, m = int(a + b), int(c + d)
            if h >= 24 or m >= 60:
                continue
            gap = (h * 60 + m - cur) % 1440 or 1440
            if gap < best_gap:
                best_gap, best = gap, f"{h}:{m}"        # SAI: thiếu :02d
        return best

print("sai :", WrongNoPad().nextClosestTime("01:09"), "| đúng:", Solution().nextClosestTime("01:09"))
print("sai :", WrongNoPad().nextClosestTime("00:00"), "| đúng:", Solution().nextClosestTime("00:00"))

sai : 1:10 | đúng: 01:10
sai : 0:0 | đúng: 00:00


**Vì sao sai:** `f"{h}:{m}"` in `h=1, m=0` thành `"1:0"` chứ không phải `"01:00"`; đề yêu cầu đúng định dạng `HH:MM`. **Cách sửa:** `f"{h:02d}:{m:02d}"`.

## 14. Mẹo & nhận dạng pattern

**Dấu hiệu nhận ra bài "mốc kế tiếp trên vòng tròn":**

- Giờ, ngày trong tuần, vị trí trên vòng — mọi thứ **quay vòng** ⇒ dùng **modulo** và nhớ rằng "kế tiếp" nghĩa là khoảng cách **dương** (nên `0` đổi thành chu kỳ đầy đủ).
- "Dùng lại chữ số cho phép" với số vị trí rất ít (4) ⇒ **liệt kê vét cạn** (`product`) luôn khả thi; chỉ cần cẩn thận kiểm tra ràng buộc từng vị trí.
- Khi có thể sắp tập ứng viên: `sorted` + `bisect_right` cho "phần tử kế tiếp" và `times[0]` cho "quay về đầu".

**Mẹo cài đặt:**

- Đổi thời gian ra **số phút** để so sánh bằng phép trừ số nguyên thay vì so sánh chuỗi.
- Định dạng đầu ra luôn bằng `:02d`.
- Với bài cho đầu vào nhỏ cố định, hãy **kiểm toàn bộ không gian đầu vào** (ở đây 1440 giờ) bằng oracle chạy từng bước — thuyết phục hơn nhiều test lẻ.

**Họ bài liên quan:**

| Bài | Ý tưởng |
|---|---|
| 31. Next Permutation | "tăng vị trí phải nhất có thể tăng, đuôi về nhỏ nhất" — cùng tinh thần cách 4 |
| 556. Next Greater Element III | cùng ý tưởng hoán vị kế tiếp trên chữ số |
| 1185. Day of the Week | quay vòng theo modulo 7 |
| 401. Binary Watch | liệt kê giờ theo số bit bật |
| 539. Minimum Time Difference | khoảng cách nhỏ nhất giữa hai giờ trên vòng 1440 |

## 15. Biến thể

### 15.1. Giờ gần nhất **trước** (Previous Closest Time)

Đảo chiều: tìm giờ hợp lệ **gần nhất trước** thời điểm hiện tại (khoảng cách ngược chiều kim đồng hồ, dương; khoảng cách `0` đổi thành `1440`). Với danh sách `times` đã sắp, dùng `bisect_left(times, cur)`: phần tử ngay **bên trái** là kết quả, nếu không có thì lấy phần tử **cuối** (hôm trước).

In [26]:
from bisect import bisect_left

def previous_closest_time(time):
    digits = set(time) - {":"}
    cur = to_minutes(time)
    pairs = {a + b for a, b in product(digits, repeat=2)}
    times = sorted(int(h) * 60 + int(m) for h in pairs if int(h) < 24 for m in pairs if int(m) < 60)
    i = bisect_left(times, cur)
    prev = times[i - 1] if i > 0 else times[-1]
    return to_str(prev)

def previous_closest_time_bruteforce(time):
    allowed = set(time.replace(":", ""))
    cur = to_minutes(time)
    for step in range(1, 1441):
        s = to_str((cur - step) % 1440)
        if set(s.replace(":", "")) <= allowed:
            return s

for s in all_times:
    assert previous_closest_time(s) == previous_closest_time_bruteforce(s), s
print("19:34 ->", previous_closest_time("19:34"), "| 00:00 ->", previous_closest_time("00:00"), "| 11:11 ->", previous_closest_time("11:11"), "| 01:32 ->", previous_closest_time("01:32"))
print("đúng trên cả 1440 giờ")

19:34 -> 19:33 | 00:00 -> 00:00 | 11:11 -> 11:11 | 01:32 -> 01:31
đúng trên cả 1440 giờ


### 15.2. Định dạng có giây `HH:MM:SS`

Mở rộng tự nhiên: 6 chữ số, mốc trong ngày là `86400` giây. Cách `product(digits, repeat=6)` có tối đa `4^6 = 4096` tổ hợp (nếu 4 chữ số khác nhau) — vẫn nhỏ. Kiểm bằng oracle chạy từng giây trên một số giờ ngẫu nhiên.

In [27]:
def next_time_hms(time):
    digits = set(time) - {":"}
    cur = int(time[:2]) * 3600 + int(time[3:5]) * 60 + int(time[6:])
    best_gap, best = 86401, time
    for p in product(digits, repeat=6):
        h, m, s = int("".join(p[:2])), int("".join(p[2:4])), int("".join(p[4:]))
        if h >= 24 or m >= 60 or s >= 60:
            continue
        gap = (h * 3600 + m * 60 + s - cur) % 86400 or 86400
        if gap < best_gap:
            best_gap, best = gap, f"{h:02d}:{m:02d}:{s:02d}"
    return best

def next_time_hms_bruteforce(time):
    allowed = set(time.replace(":", ""))
    cur = int(time[:2]) * 3600 + int(time[3:5]) * 60 + int(time[6:])
    for step in range(1, 86401):
        t = (cur + step) % 86400
        h, rem = divmod(t, 3600)
        m, s = divmod(rem, 60)
        string = f"{h:02d}:{m:02d}:{s:02d}"
        if set(string.replace(":", "")) <= allowed:
            return string

rng = random.Random(15)
for _ in range(15):
    t = f"{rng.randint(0, 23):02d}:{rng.randint(0, 59):02d}:{rng.randint(0, 59):02d}"
    assert next_time_hms(t) == next_time_hms_bruteforce(t), t
print("12:34:56 ->", next_time_hms("12:34:56"), "| 23:59:59 ->", next_time_hms("23:59:59"), "| 01:01:01 ->", next_time_hms("01:01:01"))

12:34:56 -> 12:35:11 | 23:59:59 -> 22:22:22 | 01:01:01 -> 01:01:10


### 15.3. Đếm số giờ hợp lệ

Hỏi: với tập chữ số `D`, **có bao nhiêu giờ hợp lệ** (chỉ dùng chữ số trong `D`)? Đây là `|H| × |M|` với `H`, `M` là tập giờ/phút hợp lệ (cách 5). Kiểm bằng cách đếm từng phút.

In [28]:
def count_valid_times(digits):
    pairs = {a + b for a, b in product(digits, repeat=2)}
    hours = [p for p in pairs if int(p) < 24]
    minutes = [p for p in pairs if int(p) < 60]
    return len(hours) * len(minutes)

rng = random.Random(153)
for _ in range(100):
    digits = set(rng.sample("0123456789", rng.randint(1, 5)))
    direct = sum(1 for m in range(1440) if set(to_str(m).replace(":", "")) <= digits)
    assert count_valid_times(digits) == direct, digits
print("đếm công thức = đếm trực tiếp trên 100 tập chữ số ngẫu nhiên: OK")
print("tập {1,9,3,4} có", count_valid_times(set("1934")), "giờ hợp lệ;  tập {0,1,2,3,4,5} có", count_valid_times(set("012345")))

đếm công thức = đếm trực tiếp trên 100 tập chữ số ngẫu nhiên: OK
tập {1,9,3,4} có 48 giờ hợp lệ;  tập {0,1,2,3,4,5} có 576


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Chạy từng phút | thử 1, 2, ..., 1440 phút sau, kiểm tập chữ số | `O(1440)` | `O(1)` | Làm **oracle**, ý tưởng đơn giản nhất |
| 2. `product` + khoảng cách vòng | liệt kê `4^4` tổ hợp, chọn `(t - cur) % 1440` nhỏ nhất (`0 → 1440`) | `O(1)` (≤256) | `O(1)` | Ngắn gọn, ít lỗi |
| 3. DFS có cắt tỉa | dựng từng chữ số, bỏ sớm giờ/phút sai | `O(1)` (≤341 nút) | `O(L)` | Khi số vị trí lớn hơn, cần cắt tỉa |
| 4. Tham lam từ phải sang trái | tăng vị trí phải nhất có thể, đuôi về nhỏ nhất | `O(1)` (≤4 lần quét) | `O(1)` | Nhanh nhất; dễ sai ngưỡng |
| 5. Giờ × phút + `bisect` | tập giờ × tập phút, sắp xếp, `bisect_right` | `O(1)` (≤256 mốc) | `O(1)` | **Nộp bài**: không ngưỡng ma thuật |

Cả năm cách đều `O(1)` với đầu vào cố định; khác nhau ở độ gọn, độ dễ sai và khả năng mở rộng (giây, nhiều chữ số hơn).

## 17. Bài tập vận dụng

Làm lần lượt từ dễ đến khó. Các cell `TODO` chỉ **định nghĩa** hàm, không tự gọi, để notebook luôn chạy sạch.

### Bài 1 (đoán output)

Không chạy máy, dự đoán rồi đối chiếu đáp án ở mục 19:

```python
Solution().nextClosestTime("01:32")
Solution().nextClosestTime("19:34")
Solution().nextClosestTime("23:59")
Solution().nextClosestTime("11:11")
```

### Bài 2 (tính tay)

Với `time = "13:55"`: viết tập chữ số cho phép, rồi chạy **cách tham lam** từng bước (phút đơn vị, phút chục, giờ đơn vị, giờ chục) để tìm đáp án.

### Bài 3 (điền TODO)

Hoàn thành hai hàm tiện ích `to_minutes_todo` (đổi `"HH:MM"` ra số phút) và `gap_todo` (khoảng cách vòng dương từ phút `a` tới phút `b`, `0` tính là `1440`).

In [29]:
def to_minutes_todo(t):
    # TODO: trả về số phút kể từ 00:00 của chuỗi "HH:MM"
    return None

def gap_todo(a, b):
    # TODO: số phút phải chờ từ phút a tới phút b theo chiều thời gian, trong khoảng [1, 1440]
    return None
# Không gọi các hàm này ở đây. Hãy sửa rồi tự gọi ở một cell mới.

### Bài 4 (tự viết lại)

Không nhìn lại lời giải, hãy viết `next_time_bisect(time)` theo **Cách 5**: sinh tập giờ hợp lệ × tập phút hợp lệ, sắp xếp, dùng `bisect_right` và quay về đầu khi hết.

In [30]:
def next_time_bisect(time):
    # TODO: tự viết
    return None
# Không gọi hàm ở đây.

### Bài 5 (mini challenge)

Viết `next_time_strictly_different(time)` trả về giờ hợp lệ gần nhất sau hiện tại **mà khác chính giờ hiện tại**, nếu tồn tại; nếu chỉ ghép được đúng một giờ (như `11:11`) thì trả `None`. Kiểm chứng với oracle chạy từng phút.

In [31]:
def next_time_strictly_different(time):
    # TODO: tự viết
    return None
# Không gọi hàm ở đây.

## 18. Gợi ý 3 mức

**Bài 1**

- Mức 1: viết tập chữ số của mỗi giờ, rồi thử tăng chữ số cuối.
- Mức 2: `"01:32"` có chữ số `{0,1,2,3}`: chữ số cuối `2` có thể tăng lên `3`. Còn `"11:11"` chỉ có chữ số `1`.
- Mức 3: đáp án `"01:33"`, `"19:39"`, `"22:22"`, `"11:11"`.

**Bài 2**

- Mức 1: chữ số cho phép `{1, 3, 5}`.
- Mức 2: phút đơn vị `5`: có chữ số lớn hơn `5` không? Phút chục `5` (giới hạn `<= 5`) có chữ số lớn hơn không? Giờ đơn vị `3` (giờ chục là `1` nên giới hạn `<= 9`): chữ số lớn hơn `3` là `5`.
- Mức 3: đặt giờ đơn vị `5`, hai phút về chữ số nhỏ nhất `1` ⇒ `15:11`.

**Bài 3**

- Mức 1: `int(t[:2]) * 60 + int(t[3:])`.
- Mức 2: khoảng cách vòng là `(b - a) % 1440`.
- Mức 3: `return (b - a) % 1440 or 1440`.

**Bài 4**

- Mức 1: giờ hợp lệ là hai chữ số `< 24`; phút hợp lệ là hai chữ số `< 60`.
- Mức 2: dùng `product(digits, repeat=2)` cho cả hai, ghép `h * 60 + m` rồi `sorted`.
- Mức 3: `i = bisect_right(times, cur)`; `times[i] if i < len(times) else times[0]`.

**Bài 5**

- Mức 1: chỉ khác `Solution` ở chỗ bỏ chính giờ hiện tại khỏi tập ứng viên.
- Mức 2: nếu sau khi bỏ mà tập rỗng thì trả `None`.
- Mức 3: dùng `times = sorted(set(times) - {cur})`, rồi `bisect_right` như cũ.

## 19. Đáp án tham khảo

Chạy kiểm tra bằng `assert`.

In [32]:
# Bài 1
assert Solution().nextClosestTime("01:32") == "01:33"
assert Solution().nextClosestTime("19:34") == "19:39"
assert Solution().nextClosestTime("23:59") == "22:22"
assert Solution().nextClosestTime("11:11") == "11:11"
print("Bài 1: 01:33, 19:39, 22:22, 11:11")

# Bài 2
print("Bài 2: 13:55 ->", trace_greedy("13:55"))
assert Solution().nextClosestTime("13:55") == "15:11"

Bài 1: 01:33, 19:39, 22:22, 11:11
time = 13:55 | chữ số cho phép = ['1', '3', '5'] | nhỏ nhất = '1'
  xét phút đơn vị: hiện '5', giới hạn <= '9', chữ số lớn hơn: None
  xét phút chục  : hiện '5', giới hạn <= '5', chữ số lớn hơn: None
  xét giờ đơn vị : hiện '3', giới hạn <= '9', chữ số lớn hơn: '5'
  -> đặt '5' ở giờ đơn vị, các vị trí bên phải về '1': 15:11
Bài 2: 13:55 -> 15:11


In [33]:
# Bài 3
def to_minutes_answer(t):
    return int(t[:2]) * 60 + int(t[3:])

def gap_answer(a, b):
    return (b - a) % 1440 or 1440

# Bài 4
def next_time_bisect_answer(time):
    digits = set(time) - {":"}
    cur = int(time[:2]) * 60 + int(time[3:])
    pairs = {a + b for a, b in product(digits, repeat=2)}
    hours = [int(p) for p in pairs if int(p) < 24]
    minutes = [int(p) for p in pairs if int(p) < 60]
    times = sorted(h * 60 + m for h in hours for m in minutes)
    i = bisect_right(times, cur)
    nxt = times[i] if i < len(times) else times[0]
    return f"{nxt // 60:02d}:{nxt % 60:02d}"

# Bài 5
def next_time_strictly_different_answer(time):
    digits = set(time) - {":"}
    cur = to_minutes(time)
    pairs = {a + b for a, b in product(digits, repeat=2)}
    times = sorted({int(h) * 60 + int(m) for h in pairs if int(h) < 24 for m in pairs if int(m) < 60} - {cur})
    if not times:
        return None
    i = bisect_right(times, cur)
    return to_str(times[i] if i < len(times) else times[0])

def next_time_strictly_different_bruteforce(time):
    allowed = set(time.replace(":", ""))
    cur = to_minutes(time)
    for step in range(1, 1440):                      # bỏ step = 1440 (chính nó)
        s = to_str((cur + step) % 1440)
        if set(s.replace(":", "")) <= allowed:
            return s
    return None

for s in all_times:
    assert to_minutes_answer(s) == to_minutes(s)
    assert next_time_bisect_answer(s) == Solution().nextClosestTime(s)
    assert next_time_strictly_different_answer(s) == next_time_strictly_different_bruteforce(s), s
for a, b in [(0, 0), (0, 1), (1439, 0), (100, 99)]:
    assert gap_answer(a, b) == clockwise_gap(a, b)
print("Bài 3, 4, 5: OK")
print("ví dụ Bài 5: 19:34 ->", next_time_strictly_different_answer("19:34"), "| 11:11 ->", next_time_strictly_different_answer("11:11"), "| 23:59 ->", next_time_strictly_different_answer("23:59"))

Bài 3, 4, 5: OK
ví dụ Bài 5: 19:34 -> 19:39 | 11:11 -> None | 23:59 -> 22:22


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template (tập giờ × tập phút, sắp xếp, tìm nhị phân):**

```python
from bisect import bisect_right
from itertools import product

class Solution:
    def nextClosestTime(self, time: str) -> str:
        digits = set(time) - {":"}                          # chữ số cho phép
        cur = int(time[:2]) * 60 + int(time[3:])            # đổi ra số phút
        pairs = {a + b for a, b in product(digits, repeat=2)}
        hours = [int(p) for p in pairs if int(p) < 24]      # giờ hợp lệ
        minutes = [int(p) for p in pairs if int(p) < 60]    # phút hợp lệ
        times = sorted(h * 60 + m for h in hours for m in minutes)
        i = bisect_right(times, cur)                        # mốc lớn hơn hẳn cur
        nxt = times[i] if i < len(times) else times[0]      # hết thì quay về đầu (ngày mai)
        return f"{nxt // 60:02d}:{nxt % 60:02d}"            # nhớ đệm số 0
```

**Câu thần chú:** *"Đổi ra phút; giờ < 24, phút < 60; lớn hơn hẳn thì lấy, hết thì quay về đầu; in `:02d`."*

**Checklist trước khi submit:**

- [ ] "Kế tiếp" nghĩa là **sau** hiện tại (khoảng cách dương); khoảng cách `0` được tính là `1440`.
- [ ] Có xử lý **vòng qua nửa đêm** (quay về giờ nhỏ nhất của ngày mai).
- [ ] Mọi giờ ghép ra đều kiểm `h < 24` và `m < 60`.
- [ ] Đầu ra định dạng đúng `HH:MM` (đệm `0`).
- [ ] Trường hợp chỉ có một chữ số (`11:11`, `00:00`) trả lại chính giờ đó.
- [ ] Đã thử: `23:59`, `19:59`, `00:00`, giờ có chữ số `0`.
- [ ] Đã đối chiếu oracle chạy từng phút trên cả 1440 giờ.
- [ ] Độ phức tạp: `O(1)` (đầu vào cố định 5 ký tự), hằng số nhỏ.
- [ ] Đây là bài **Premium**: đã đối chiếu lại đề gốc nếu có tài khoản.